In [32]:
from decimal import Decimal, ROUND_HALF_UP
import math

In [33]:
def formatear(x, cifras):
    """Notación científica si |x| < 1 (como en papel), decimal normal si no."""
    if x is None:
        return "-"
    if x != 0 and abs(x) < 1:
        return a_cientifica(x, cifras)
    return f"{x:.{cifras}f}"


def imprimir_tabla(filas, cifras):
    encabezado = (f"{'n':<5}{'x_n':^25}{'f(x_n)':^25}"
                  f"{'f\'(x_n)':^25}{'Error':^25}")
    print(encabezado)
    print("-" * len(encabezado))
    for n, x, fx, dfx, error in filas:
        print(f"{n:<5}"
              f"{formatear(x, cifras):^25}"
              f"{formatear(fx, cifras):^25}"
              f"{formatear(dfx, cifras):^25}"
              f"{formatear(error, cifras):^25}")

def imprimir_tabla1(filas, cifras):
    encabezado = (f"{'n':<5}{'x_n':^25}{'f(x_n)':^25}"
                  f"{'Error':^25}")
    print(encabezado)
    print("-" * len(encabezado))
    for n, x, fx, error in filas:
        print(f"{n:<5}"
              f"{formatear(x, cifras):^25}"
              f"{formatear(fx, cifras):^25}"
              f"{formatear(error, cifras):^25}")
        
def imprimir_tabla2(filas, cifras):
    encabezado = (f"{'n':<5}{'p_n':^18}{'f(p_n)':^18}{'sgm(f(p_n))':^18}"
                  f"{'Error':^18}")
    print(encabezado)
    print("-" * len(encabezado))
    for n, p_n, fp_n, sgm, error in filas:
        print(f"{n:<5}"
              f"{formatear(p_n, cifras):^18}"
              f"{formatear(fp_n, cifras):^18}"
              f"{formatear(sgm, cifras):^18}"
              f"{formatear(error, cifras):^18}")

In [34]:
def metodo_newton(f, df, p0, TOL, n_max, cifras=6):
    """
    Calcula una aproximación a la raíz de una función mediante el método de Newton-Raphson,
    aplicando redondeo en cada iteración para controlar la precisión de los cálculos.

    parameters:
        f (function): Función cuya raíz se desea encontrar.
        df (function): Derivada de la función f.
        p0 (float): Aproximación inicial a la raíz.
        TOL (float): Tolerancia máxima permitida para el error entre aproximaciones sucesivas.
        n_max (int): Número máximo de iteraciones permitidas.
        cifras (int): Cantidad de decimales utilizados en el redondeo. Por defecto, es 6.

    returns:
        float: Aproximación a la raíz si el procedimiento es exitoso.
        int: -1 si la derivada es cero o si no se alcanza la tolerancia dentro del número
             máximo de iteraciones.
    """
    i = 1

    p0 = redondear(p0, cifras)
    fx = redondear(f(p0), cifras)
    dfx = redondear(df(p0), cifras)

    filas = [(0, p0, fx, dfx, None)]

    while i <= n_max:
        if dfx == 0:
            imprimir_tabla(filas, cifras)
            print("Derivada igual a 0, causa indeterminación")
            return -1

        p = redondear(p0 - fx / dfx, cifras)
        error = redondear(abs(p - p0), cifras)

        # f(p) y f'(p) ya quedan listos para la siguiente fila/iteración
        fx = redondear(f(p), cifras)
        dfx = redondear(df(p), cifras)

        filas.append((i, p, fx, dfx, error))

        if error < TOL:
            imprimir_tabla(filas, cifras)
            print("\nProcedimiento exitoso!")
            return p

        i = i + 1
        p0 = p

    imprimir_tabla(filas, cifras)
    print(f"\nEl procedimiento falló después de {n_max} iteraciones")
    return -1
    

In [35]:
def redondear(x, cifras):
    """
    Emula la calculadora:
      - Si |x| >= 1: redondea a `cifras` decimales (half-up).
      - Si |x| < 1: pasa a notación científica y redondea la
        mantisa a `cifras` decimales (half-up).
    Devuelve un float.
    """
    d = Decimal(str(x))
    paso = Decimal(1).scaleb(-cifras)   # 0.000001 si cifras = 6

    if d == 0 or abs(d) >= 1:
        return float(d.quantize(paso, rounding=ROUND_HALF_UP))

    e = d.adjusted()                   # exponente científico
    m = (d.scaleb(-e)).quantize(paso, rounding=ROUND_HALF_UP)
    # pongo -e para que la mantisa sea facil de redondear, verlo como 1.2345

    # Caso borde: 9.9999996 -> 10.000000
    if abs(m) >= 10:
        m = m / 10
        e += 1

    return float(m.scaleb(e))
    #aqui con e, la traigo a su flotante normal 0.1234567, con 7 cifras porque una de ellas es 
    #el entero


Con d.scaleb(-e) desplazo la coma hacia la primera cifra significativa, es decir:

0.0012345 => 1.2345

Esto porque e me arroja la posicion de la primera cifra significativa, en este caso será negativo y aquí al volver a poner un -, lo que hago es desplzarme hacia la derecha.

.quantize() redondea y se encarga de multiplcar el numero por el x10^(algo) que calculaste anteriormente, con el fin de que tenga la cantidad de cifras que solicitaste.



In [36]:
def a_cientifica(x, cifras):
    d = Decimal(str(x))
    if d == 0:
        return f"{0:.{cifras}f}e+00"
    paso = Decimal(1).scaleb(-cifras)
    #print(paso)
    e = d.adjusted()
    #print(e)
    m = d.scaleb(-e).quantize(paso, rounding=ROUND_HALF_UP)
    if abs(m) >= 10:
        m = m / 10
        e += 1
    return f"{m}e{e:+03d}"

#a_cientifica(34.5667, 4)

In [37]:
raiz = metodo_newton(lambda x: x**3 + 3*x**2 -1, lambda x: 3*x**2 + 6*x, -2.5, 1e-4, 20, 8)
print("Raíz aproximada:", raiz)

n               x_n                    f(x_n)                   f'(x_n)                   Error          
---------------------------------------------------------------------------------------------------------
0           -2.50000000              2.12500000               3.75000000                    -            
1           -3.06666667              -1.62696300              9.81333337             5.66666670e-01      
2           -2.90087560            -1.65860319e-01            7.83998414             1.65791070e-01      
3           -2.87971990            -2.54278615e-03            7.60004071             2.11557000e-02      
4           -2.87938532            -5.95761431e-07            7.59626754             3.34580000e-04      
5           -2.87938524            1.19399424e-08             7.59626664             8.00000000e-08      

Procedimiento exitoso!
Raíz aproximada: -2.87938524


In [38]:
metodo_newton(lambda x: x-math.cos(x), lambda x: 1 + math.sin(x), math.pi/4, 1e-4, 20, 8 )

n               x_n                    f(x_n)                   f'(x_n)                   Error          
---------------------------------------------------------------------------------------------------------
0         7.85398163e-01           7.82913815e-02             1.70710678                    -            
1         7.39536134e-01           7.54875494e-04             1.67394529             4.58620290e-02      
2         7.39085178e-01           7.49524465e-08             1.67361206             4.50956000e-04      
3         7.39085133e-01           -3.60095398e-10            1.67361203             4.49999999e-08      

Procedimiento exitoso!


0.739085133

In [39]:
metodo_newton(lambda x: x-0.8-0.2*math.sin(x), lambda x : 1-0.2*math.cos(x), math.pi/4, 1e-4, 20, 8)

n               x_n                    f(x_n)                   f'(x_n)                   Error          
---------------------------------------------------------------------------------------------------------
0         7.85398163e-01           -1.56023193e-01          8.58578644e-01                  -            
1         9.67120821e-01           2.46988139e-03           8.86465564e-01           1.81722658e-01      
2         9.64334609e-01           6.39081222e-07           8.86007253e-01           2.78621200e-03      
3         9.64333888e-01           2.70034828e-10           8.86007135e-01           7.21000000e-07      

Procedimiento exitoso!


0.964333888

In [40]:
def secante(f, p0, p1, TOL, n_max, cifras):
    i = 2
    
    p0 = redondear(p0, cifras)
    p1 = redondear(p1, cifras)
    fp0 = redondear(f(p0), cifras)
    fp1 = redondear(f(p1), cifras)
    
    filas = [(0, p0, fp0, None)]
    filas.append((1, p1, fp1, None))
    
    while i <= n_max:
        
        denominador = (fp1 - fp0)
        
        if denominador == 0:
            imprimir_tabla1(filas, cifras)
            print("La resta entre fp1 y fp0 genera indeterminación")
            return -1
        
        p = redondear(p1 - fp1*(p1 - p0)/denominador, cifras)
        error = redondear(abs(p - p1), cifras)
        fp = redondear(f(p), cifras) #lo calculo aqui para poder mostrarlo
        
        filas.append((i, p, fp, error))
        
        if error < TOL:
            imprimir_tabla1(filas, cifras)
            print("Procedimiento Exitoso")
            return p
        
        
        p0  = p1
        fp0 = fp1
        p1  = p
        fp1 = fp
        
        i = i +1
    
    imprimir_tabla1(filas, cifras)
    
    print(f"\nEl procedimiento falló después de {n_max} iteraciones")
    return -1

In [41]:
def signum(a):
    if a < 0:
        return -1
    elif a > 0:
        return 1
    else:
        return 0

In [42]:
def posicion_falsa(f, p0, p1, TOL, n_max, cifras):
    
    i = 2
    
    p0 = redondear(p0, cifras)
    p1 = redondear(p1, cifras)
    
    fp0 = redondear(f(p0), cifras)
    sgm0 = signum(fp0)
    
    fp1 = redondear(f(p1), cifras)
    sgm1 = signum(fp1)
    
    if sgm0 * sgm1 > 0:
        print("f(p0) y f(p1) tienen el mismo signo: el intervalo no encierra una raíz")
        return -1
    
    filas = [(0, p0, fp0, sgm0, None)]
    filas.append((1, p1, fp1, sgm1, None))
    
    while i <= n_max:
        
        denominador = (fp1 - fp0)
                
        if denominador == 0:
            imprimir_tabla2(filas, cifras)
            print("La resta entre fp1 y fp0 genera indeterminación")
            return -1
        
        p = redondear(p1 - fp1 * (p1 - p0)/(denominador), cifras)
        error = redondear(abs(p - p1), cifras)
        
        fp  = redondear(f(p), cifras)
        sgmfp = signum(fp)
        
        filas.append((i, p, fp, sgmfp, error))
        
        
        if error <= TOL:
            imprimir_tabla2(filas, cifras)
            print("Procedimiento Exitoso")
            return p
        
        i = i + 1
        
        if sgmfp * sgm1 < 0:
            p0  = p1
            fp0 = fp1
        
        p1  = p
        fp1 = fp
        sgm1 = sgmfp
    
    imprimir_tabla2(filas, cifras)
    
    print(f"\nEl procedimiento falló después de {n_max} iteraciones")
    return -1
        

In [43]:
secante(lambda x : math.cos(x) - x, 0.5, math.pi/4, 1e-16, 50, 13)

n               x_n                    f(x_n)                    Error          
--------------------------------------------------------------------------------
0       5.0000000000000e-01      3.7758256189037e-01               -            
1       7.8539816339745e-01     -7.8291382210904e-02               -            
2       7.3638413883658e-01      4.5177185221738e-03      4.9014024560870e-02   
3       7.3905813921389e-01      4.5177215963310e-05      2.6740003773099e-03   
4       7.3908514933728e-01     -2.6982172940393e-08      2.7010123390014e-05   
5       7.3908513321506e-01      1.6853185513810e-13      1.6122220003467e-08   
6       7.3908513321516e-01      1.1102230246252e-15      1.0003109451873e-13   
7       7.3908513321516e-01      1.1102230246252e-15        0.0000000000000     
Procedimiento Exitoso


0.73908513321516

In [44]:
metodo_newton(lambda x : math.cos(x) - x, lambda x : -math.sin(x) - 1, (0.5+(math.pi/4))/2, 1e-16, 20, 13)

n               x_n                    f(x_n)                   f'(x_n)                   Error          
---------------------------------------------------------------------------------------------------------
0       6.4269908169872e-01      1.5778187720746e-01       -1.5993581854189                 -            
1       7.4135232813591e-01     -3.7963028953547e-03       -1.6752859465673        9.8653246437190e-02   
2       7.3908626549560e-01     -1.8949986374528e-06       -1.6736128660344        2.2660626403100e-03   
3       7.3908513321544e-01     -4.6751491566965e-13       -1.6736120291834        1.1322801599656e-06   
4       7.3908513321516e-01      1.1102230246252e-15       -1.6736120291832        2.7999824681046e-13   
5       7.3908513321516e-01      1.1102230246252e-15       -1.6736120291832          0.0000000000000     

Procedimiento exitoso!


0.73908513321516

In [45]:
metodo_newton(lambda x: 3*x - math.exp(x), lambda x : 3 - math.exp(x), 1.5, 1e-5, 20, 5)

n               x_n                    f(x_n)                   f'(x_n)                   Error          
---------------------------------------------------------------------------------------------------------
0             1.50000                1.83109e-02               -1.48169                     -            
1             1.51236               -3.46495e-04               -1.53743                1.23600e-02       
2             1.51213                6.99314e-06               -1.53638                2.30000e-04       
3             1.51213                6.99314e-06               -1.53638                  0.00000         

Procedimiento exitoso!


1.51213

In [46]:
secante(lambda x: 3*x - math.exp(x), 1, 2, 1e-5, 20, 9)

n               x_n                    f(x_n)                    Error          
--------------------------------------------------------------------------------
0           1.000000000            2.817181715e-01                 -            
1           2.000000000             -1.389056099                   -            
2           1.168615340            2.883116585e-01          8.313846600e-01     
3           1.311516555            2.227510724e-01          1.429012150e-01     
4           1.797043009           -6.406561067e-01          4.855264540e-01     
5           1.436777893            1.032155095e-01          3.602651160e-01     
6           1.486766287            3.752846131e-02          4.998839400e-02     
7           1.515325760           -4.926107689e-03          2.855947300e-02     
8           1.512011934            1.883561164e-04          3.313826000e-03     
9           1.512133976            8.844420609e-07          1.220420000e-04     
10          1.512134552     

1.512134552

In [47]:
posicion_falsa(lambda x: 3*x - math.exp(x), 1, 2, 1e-5, 20, 9)

n           p_n              f(p_n)         sgm(f(p_n))          Error       
-----------------------------------------------------------------------------
0       1.000000000     2.817181715e-01     1.000000000            -         
1       2.000000000       -1.389056099      -1.000000000           -         
2       1.168615340     2.883116585e-01     1.000000000     8.313846600e-01  
3       1.311516555     2.227510724e-01     1.000000000     1.429012150e-01  
4       1.406664677     1.376772040e-01     1.000000000     9.514812200e-02  
5       1.460170255     7.381806310e-02     1.000000000     5.350557800e-02  
6       1.487410593     3.661084360e-02     1.000000000     2.724033800e-02  
7       1.500573787     1.746001784e-02     1.000000000     1.316319400e-02  
8       1.506773496     8.171671645e-03     1.000000000     6.199709000e-03  
9       1.509658126     3.790890742e-03     1.000000000     2.884630000e-03  
10      1.510992682     1.751416409e-03     1.000000000     1.33

1.51212954

Método de Newton Ejercicios 1 y 2

In [48]:
metodo_newton(lambda x: x**2 - 6, lambda x: 2*x, 1, 1e-3, 20, 4)

n               x_n                    f(x_n)                   f'(x_n)                   Error          
---------------------------------------------------------------------------------------------------------
0             1.0000                   -5.0000                  2.0000                      -            
1             3.5000                   6.2500                   7.0000                   2.5000          
2             2.6071                 7.9697e-01                 5.2142                 8.9290e-01        
3             2.4543                 2.3588e-02                 4.9086                 1.5280e-01        
4             2.4495                 5.0250e-05                 4.8990                 4.8000e-03        
5             2.4495                 5.0250e-05                 4.8990                   0.0000          

Procedimiento exitoso!


2.4495

In [49]:
metodo_newton(lambda x: -x**3 - math.cos(x), lambda x: -3*x**2 + math.sin(x), -1, 1e-3, 20, 4)

n               x_n                    f(x_n)                   f'(x_n)                   Error          
---------------------------------------------------------------------------------------------------------
0             -1.0000                4.5970e-01                 -3.8415                     -            
1           -8.8033e-01              4.5342e-02                 -3.0959                1.1967e-01        
2           -8.6568e-01              6.1978e-04                 -3.0097                1.4650e-02        
3           -8.6547e-01              -1.2134e-05                -3.0085                2.1000e-04        

Procedimiento exitoso!


-0.86547

Ejercicio 3, literales a,b,c,d.

In [50]:
metodo_newton(lambda x: x**3 - 2*x**2 - 5, lambda x: 3*x**2 - 4*x, 2.5, 1e-4, 20, 4)

n               x_n                    f(x_n)                   f'(x_n)                   Error          
---------------------------------------------------------------------------------------------------------
0             2.5000                   -1.8750                  8.7500                      -            
1             2.7143                 2.6255e-01                 11.2451                2.1430e-01        
2             2.6910                 3.8634e-03                 10.9604                2.3300e-02        
3             2.6906                 -5.1983e-04                10.9556                4.0000e-04        
4             2.6906                 -5.1983e-04                10.9556                  0.0000          

Procedimiento exitoso!


2.6906

In [51]:
metodo_newton(lambda x: x**3 + 3*x**2 - 1, lambda x: 3*x**2 + 6*x, -2.5, 1e-4, 20, 4)

n               x_n                    f(x_n)                   f'(x_n)                   Error          
---------------------------------------------------------------------------------------------------------
0             -2.5000                  2.1250                   3.7500                      -            
1             -3.0667                  -1.6273                  9.8137                 5.6670e-01        
2             -2.9009                -1.6605e-01                7.8403                 1.6580e-01        
3             -2.8797                -2.3915e-03                7.5998                 2.1200e-02        
4             -2.8794                -1.1211e-04                7.5964                 3.0000e-04        
5             -2.8794                -1.1211e-04                7.5964                   0.0000          

Procedimiento exitoso!


-2.8794

In [52]:
metodo_newton(lambda x: x - math.cos(x), lambda x: 1 + math.sin(x), math.pi/4, 1e-4, 20, 4)

n               x_n                    f(x_n)                   f'(x_n)                   Error          
---------------------------------------------------------------------------------------------------------
0           7.8540e-01               7.8295e-02                 1.7071                      -            
1           7.3954e-01               7.6135e-04                 1.6739                 4.5860e-02        
2           7.3909e-01               8.1451e-06                 1.6736                 4.5000e-04        
3           7.3909e-01               8.1451e-06                 1.6736                   0.0000          

Procedimiento exitoso!


0.73909

In [53]:
metodo_newton(lambda x: x - 0.8 - 0.2*math.sin(x), lambda x: 1 - 0.2*math.cos(x), math.pi/4, 1e-4, 20, 4)

n               x_n                    f(x_n)                   f'(x_n)                   Error          
---------------------------------------------------------------------------------------------------------
0           7.8540e-01               -1.5602e-01              8.5858e-01                    -            
1           9.6712e-01               2.4692e-03               8.8647e-01               1.8172e-01        
2           9.6433e-01               -3.4445e-06              8.8601e-01               2.7900e-03        
3           9.6433e-01               -3.4445e-06              8.8601e-01                 0.0000          

Procedimiento exitoso!


0.96433

Ejercicio 3 a

In [54]:
metodo_newton(lambda x : 3*x - math.exp(x), lambda x : 3 - math.exp(x), 1.5, 1e-5, 20, 4)

n               x_n                    f(x_n)                   f'(x_n)                   Error          
---------------------------------------------------------------------------------------------------------
0             1.5000                 1.8311e-02                 -1.4817                     -            
1             1.5124                 -4.0800e-04                -1.5376                1.2400e-02        
2             1.5121                 5.3083e-05                 -1.5362                3.0000e-04        
3             1.5121                 5.3083e-05                 -1.5362                  0.0000          

Procedimiento exitoso!


1.5121

In [55]:
secante(lambda x : 3*x - math.exp(x), 1, 2, 1e-5, 20, 4)

n               x_n                    f(x_n)                    Error          
--------------------------------------------------------------------------------
0             1.0000                 2.8172e-01                    -            
1             2.0000                   -1.3891                     -            
2             1.1686                 2.8831e-01               8.3140e-01        
3             1.3115                 2.2276e-01               1.4290e-01        
4             1.7971                 -6.4083e-01              4.8560e-01        
5             1.4368                 1.0319e-01               3.6030e-01        
6             1.4868                 3.7480e-02               5.0000e-02        
7             1.5153                 -4.8862e-03              2.8500e-02        
8             1.5120                 2.0668e-04               3.3000e-03        
9             1.5121                 5.3083e-05               1.0000e-04        
10            1.5121        

1.5121

In [56]:
posicion_falsa(lambda x : 3*x - math.exp(x), 1, 2, 1e-5, 20, 4)

n           p_n              f(p_n)         sgm(f(p_n))          Error       
-----------------------------------------------------------------------------
0          1.0000          2.8172e-01          1.0000              -         
1          2.0000           -1.3891           -1.0000              -         
2          1.1686          2.8831e-01          1.0000          8.3140e-01    
3          1.3115          2.2276e-01          1.0000          1.4290e-01    
4          1.4067          1.3764e-01          1.0000          9.5200e-02    
5          1.4602          7.3779e-02          1.0000          5.3500e-02    
6          1.4874          3.6626e-02          1.0000          2.7200e-02    
7          1.5006          1.7421e-02          1.0000          1.3200e-02    
8          1.5068          8.1316e-03          1.0000          6.2000e-03    
9          1.5097          3.7270e-03          1.0000          2.9000e-03    
10         1.5110          1.7402e-03          1.0000          1

1.5121

Ejercicio 4 b

In [57]:
metodo_newton(lambda x : 2*x + 3* math.cos(x) - math.exp(x), lambda x : 2-3*math.sin(x) - math.exp(x), 1.5, 1e-5, 20, 4 )

n               x_n                    f(x_n)                   f'(x_n)                   Error          
---------------------------------------------------------------------------------------------------------
0             1.5000                   -1.2695                  -5.4742                     -            
1             1.2681                 -1.2361e-01                -4.4177                2.3190e-01        
2             1.2401                 -1.6539e-03                -4.2934                2.8000e-02        
3             1.2397                 6.3079e-05                 -4.2916                4.0000e-04        
4             1.2397                 6.3079e-05                 -4.2916                  0.0000          

Procedimiento exitoso!


1.2397

In [60]:
secante(lambda x : 2*x + 3* math.cos(x) - math.exp(x), 1, 2, 1e-5, 20, 4)

n               x_n                    f(x_n)                    Error          
--------------------------------------------------------------------------------
0             1.0000                 9.0263e-01                    -            
1             2.0000                   -4.6375                     -            
2             1.1629                 3.1664e-01               8.3710e-01        
3             1.2164                 9.8857e-02               5.3500e-02        
4             1.2407                 -4.2308e-03              2.4300e-02        
5             1.2397                 6.3079e-05               1.0000e-03        
6             1.2397                 6.3079e-05                 0.0000          
Procedimiento Exitoso


1.2397

In [61]:
posicion_falsa(lambda x : 2*x + 3* math.cos(x) - math.exp(x), 1, 2, 1e-5, 20, 4)

n           p_n              f(p_n)         sgm(f(p_n))          Error       
-----------------------------------------------------------------------------
0          1.0000          9.0263e-01          1.0000              -         
1          2.0000           -4.6375           -1.0000              -         
2          1.1629          3.1664e-01          1.0000          8.3710e-01    
3          1.2164          9.8857e-02          1.0000          5.3500e-02    
4          1.2328          2.9570e-02          1.0000          1.6400e-02    
5          1.2377          8.6375e-03          1.0000          4.9000e-03    
6          1.2391          2.6373e-03          1.0000          1.4000e-03    
7          1.2395          9.2132e-04          1.0000          4.0000e-04    
8          1.2397          6.3079e-05          1.0000          2.0000e-04    
9          1.2397          6.3079e-05          1.0000            0.0000      
Procedimiento Exitoso


1.2397

In [64]:
posicion_falsa(lambda x: 230*x**4 + 18*x**3 + 9*x**2 -221 * x -9, -1, 0, 1e-6, 20, 4 )

n           p_n              f(p_n)         sgm(f(p_n))          Error       
-----------------------------------------------------------------------------
0         -1.0000           433.0000           1.0000              -         
1          0.0000           -9.0000           -1.0000              -         
2       -2.0362e-02         -4.4964           -1.0000          2.0362e-02    
3       -3.0430e-02         -2.2669           -1.0000          1.0068e-02    
4       -3.5480e-02         -1.1480           -1.0000          5.0500e-03    
5       -3.8030e-02       -5.8286e-01         -1.0000          2.5500e-03    
6       -3.9323e-02       -2.9624e-01         -1.0000          1.2930e-03    
7       -3.9980e-02       -1.5060e-01         -1.0000          6.5700e-04    
8       -4.0314e-02       -7.6551e-02         -1.0000          3.3400e-04    
9       -4.0484e-02       -3.8862e-02         -1.0000          1.7000e-04    
10      -4.0570e-02       -1.9796e-02         -1.0000          8

-0.040659

In [69]:
secante(lambda x: 230*x**4 + 18*x**3 + 9*x**2 -221 * x -9, -1, 0, 1e-6, 20, 4 )

n               x_n                    f(x_n)                    Error          
--------------------------------------------------------------------------------
0             -1.0000                 433.0000                     -            
1             0.0000                   -9.0000                     -            
2           -2.0362e-02                -4.4964                2.0362e-02        
3           -4.0691e-02              7.0306e-03               2.0329e-02        
4           -4.0659e-02              -6.3921e-05              3.2000e-05        
5           -4.0659e-02              -6.3921e-05                0.0000          
Procedimiento Exitoso


-0.040659